# ULTRA-DETERMINISTIC CLOUD WORKFLOW ENGINE
### Otimizado para Kaggle (Dual T4 x2 - 32GB VRAM) e Google Colab
- **Modo de Execução:** Sandbox Estrita / Sem Dependências Locais
- **Resiliência:** Checkpoints Atômicos com Retomada Automática
- **Acesso Headless:** Túnel Cloudflared Automático sem necessidade de manter aba aberta

In [ ]:
!pip install -q torch accelerate transformers huggingface_hub fastapi uvicorn requests

In [ ]:
import os
import sys
import json
import time
import shutil
import logging
import subprocess
import threading
from typing import Dict, Any, List, Optional

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("CloudNotebookEngine")

import torch
from accelerate import Accelerator
from fastapi import FastAPI
import uvicorn

print(f"CUDA Disponível: {torch.cuda.is_available()}")
print(f"Total de GPUs: {torch.cuda.device_count()}")

In [ ]:
# -------------------------------------------------------------------------
# GERENCIADOR ATÔMICO DE ESTADO & CHECKPOINT
# -------------------------------------------------------------------------
class CheckpointEngine:
    def __init__(self, filepath="cloud_state.json"):
        self.filepath = os.path.abspath(filepath)
        self.tmp_path = f"{self.filepath}.tmp"
        self.state = self._load_or_create()

    def _load_or_create(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    logger.info(f"Estado anterior recuperado com sucesso. Itens prontos: {len(data.get('completed_tasks', []))}")
                    return data
            except Exception:
                pass
        return {
            "session_id": f"sess_{int(time.time())}",
            "completed_tasks": [],
            "stats": {"processed": 0}
        }

    def has_task(self, task_id: str) -> bool:
        return task_id in self.state["completed_tasks"]

    def save_checkpoint(self, task_id: str, metadata: Optional[Dict[str, Any]] = None):
        self.state["completed_tasks".strip()].append(task_id)
        self.state["stats"]["processed"] += 1
        with open(self.tmp_path, "w", encoding="utf-8") as f:
            json.dump(self.state, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(self.tmp_path, self.filepath)
        logger.info(f"[CHECKPOINT SALVO] Tarefa {task_id} persistida no disco.")

In [ ]:
# -------------------------------------------------------------------------
# INICIALIZAÇÃO DO TÚNEL CLOUDFLARE E TELEMETRIA
# -------------------------------------------------------------------------
checkpoint = CheckpointEngine("cloud_state.json")

# API FastAPI em Background
app = FastAPI(title="Cloud Telemetry")
@app.get("/")
def health():
    return {
        "status": "online",
        "device_count": torch.cuda.device_count(),
        "state": checkpoint.state
    }

threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()

# Download e start do Cloudflared
if not os.path.exists("cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
    !chmod +x cloudflared

def run_tunnel():
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ""):
        if "trycloudflare.com" in line:
            for tok in line.split():
                if "trycloudflare.com" in tok:
                    print(f"\n🌐 URL PÚBLICA DO TÚNEL: {tok.strip()}\n")
                    break

threading.Thread(target=run_tunnel, daemon=True).start()
time.sleep(4)

In [ ]:
# -------------------------------------------------------------------------
# EXECUÇÃO DISTRIBUÍDA MULTI-GPU (DUAL T4)
# -------------------------------------------------------------------------
accelerator = Accelerator()
device = accelerator.device
print(f"Dispositivo alocado: {device}")

tasks = [{"id": f"task_segment_{i:03d}"} for i in range(1, 21)]

for t in tasks:
    t_id = t["id"]
    if checkpoint.has_task(t_id):
        print(f"Pular {t_id} (já concluído)")
        continue

    print(f"Executando {t_id} no hardware {device}...")
    with torch.inference_mode():
        x = torch.randn((4096, 4096), device=device, dtype=torch.float16)
        y = torch.randn((4096, 4096), device=device, dtype=torch.float16)
        z = torch.matmul(x, y)
        _ = float(torch.sum(z).cpu())

    checkpoint.save_checkpoint(t_id)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✅ TODAS AS TAREFAS PROCESSADAS COM SUCESSO!")